<a href="https://colab.research.google.com/github/lakshmilokesh2205-lgtm/gamefinder/blob/main/GameFinder.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#GameFinder

My app helps gamers on a budget decide which Steam games to purchase by comparing prices, ratings, and playtime to find the best value.

In [30]:
from google.colab import drive
drive.mount("/content/drive")

import glob
import os
import time
import pandas as pd

# Look for steam_games.csv in My Drive and in folders up to 3 levels down
found = []
for pattern in ["", "*/", "*/*/", "*/*/*/"]:
    found = found + glob.glob("/content/drive/MyDrive/" + pattern + "steam_games.csv")

if len(found) == 0:
    print("❌ Can't find steam_games.csv in your Drive. Upload it next to this notebook and run this cell again.")
else:
    FOLDER = os.path.dirname(found[0]) + "/"
    games = pd.read_csv(FOLDER + "steam_games.csv")
    print(f"✅ Loaded {len(games)} games from:", FOLDER)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Loaded 1000 games from: /content/drive/MyDrive/


In [31]:
def best_positive_within_budget(games, budget):
    affordable = games[games["price"] <= budget]
    ranked = affordable.sort_values("pct_pos_total", ascending=False)
    return ranked[["name", "price", "pct_pos_total"]].head(5)

print(best_positive_within_budget(games, 20))

                          name  price  pct_pos_total
473  HoloCure - Save the Fans!   0.00             99
54           People Playground   9.99             98
63           Vampire Survivors   4.99             98
399                    OneShot   9.99             98
404             VPet-Simulator   0.00             98


In [32]:
def best_playtime(games, budget):
    affordable = games[games["price"] <= budget]
    paid = affordable[affordable["price"] > 0].copy()
    paid["hours_per_dollar"] = paid["avg_playtime_hours"] / paid["price"]
    return paid.sort_values("hours_per_dollar", ascending=False)[["name", "price", "hours_per_dollar"]].head(5)

print(best_playtime(games, 20))

                                name  price  hours_per_dollar
220                  Killing Floor 2   2.99        182.675585
418                Age of History II   4.99         70.120240
255                   Cookie Clicker   4.99         61.863727
4    Tom Clancy's Rainbow Six® Siege   3.99         59.323308
579                    Bright Memory   7.99         43.391740


In [33]:
def best_genre(games, genre, budget):
  affordable = games[games["price"] <= budget]
  matches = affordable[affordable["genres"].str.contains(genre, case=False, na=False)]
  return matches[["name", "price", "pct_pos_total"]].head(5)

print(best_genre(games, "Sports", 20))

                             name  price  pct_pos_total
28                      Fall Guys   0.00             81
30                 Rocket League®   0.00             88
62                         VRChat   0.00             75
112  theHunter: Call of the Wild™  19.99             89
137                       FIFA 22   0.00             80


In [34]:
class GameFinder:
  def __init__(self, games, budget):
     self.games = games
     self.budget = budget

  def show_games(self):
    return best_positive_within_budget(self.games, self.budget)

app = GameFinder(games, 20)
print(app.show_games())

                          name  price  pct_pos_total
473  HoloCure - Save the Fans!   0.00             99
54           People Playground   9.99             98
63           Vampire Survivors   4.99             98
399                    OneShot   9.99             98
404             VPet-Simulator   0.00             98


In [35]:
import requests

url = "https://www.cheapshark.com/api/1.0/deals"

params = {"title": "Batman", "pageSize": 5}
headers = {"User-Agent": "GameFinder/1.0 (student project)"}
response = requests.get(url, params=params, headers=headers)


deals = response.json()
print(response.status_code)


for deal in deals:
  print(deal["title"], "$" + deal["salePrice"])

200
LEGO Batman $2.99
Batman: Arkham Origins $3.99
LEGO Batman: The Videogame $2.99
Batman: Arkham Knight $3.99
LEGO Batman 2: DC Super Heroes $3.99
